# **Proyecto Integrador - Minería de Datos 1**

**1- Inspección Inicial**



**Preguntas de análisis que van a guiar el proyecto:**
1. ¿Cómo se distribuye la edad de los usuarios?
2. ¿Cómo se distribuye el tiempo de visualización mensual?
3. ¿El tiempo de visualización varía según el plan de suscripción?
4. ¿Existe relación entre la edad y la cantidad de tickets de soporte?
5. ¿El perfil de consumo por plan es consistente entre países, o varía según el país?

## Carga del dataset

In [1]:
# Importar librerías
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Carga del dataset
df = pd.read_json('/content/drive/MyDrive/streaming_users_dirty.json')

# Preservamos una copia de referencia con el total original de filas,
# útil más adelante para calcular porcentajes de retención
TOTAL_ORIGINAL = len(df)

print(f'Filas cargadas: {TOTAL_ORIGINAL}')
df.head()

Filas cargadas: 8160


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## Contexto del dataset

Cada fila representa un usuario de una plataforma de streaming. Las columnas registradas son:

| Variable | Descripción |
|---|---|
| `user_id` | Identificador único de usuario |
| `age` | Edad del usuario |
| `subscription_plan` | Plan de suscripción (Básico / Estándar / Premium) |
| `monthly_watch_time_mins` | Minutos de visualización en el último mes |
| `country` | País del usuario |
| `favorite_genre` | Género de contenido favorito |
| `last_login_date` | Fecha del último inicio de sesión |
| `customer_support_tickets` | Cantidad de tickets de soporte generados por el usuario |


## Estructura general

In [4]:
# Dimensiones del dataset
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

Dimensiones: 8160 filas x 8 columnas


In [5]:
# Tipos de datos por columna
df.dtypes

,0
user_id,int64
age,int64
subscription_plan,object
monthly_watch_time_mins,float64
country,object
favorite_genre,object
last_login_date,object
customer_support_tickets,int64


In [6]:
# Información general
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


## Valores faltantes


In [7]:
# Calculamos la cantidad y el porcentaje de valores faltantes por columna

faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(2)

resumen_faltantes = pd.DataFrame({'Faltantes': faltantes, '%': porcentaje})
print('=== Valores faltantes por columna ===')
print(resumen_faltantes.query('Faltantes > 0'))

=== Valores faltantes por columna ===
                         Faltantes     %
monthly_watch_time_mins        193  2.37
favorite_genre                 240  2.94
last_login_date                320  3.92


## Duplicados

In [10]:
# Revisamos duplicados en dos niveles: por `user_id` y por fila completa (todas las columnas idénticas)

# Duplicados en user_id (debería ser un identificador único)
duplicados_id = df['user_id'].duplicated().sum()
print(f'Registros con user_id duplicado: {duplicados_id}')

Registros con user_id duplicado: 160


In [11]:
# Filas completamente duplicadas (todas las columnas iguales)
duplicados_fila = df.duplicated().sum()
print(f'Filas completamente duplicadas: {duplicados_fila}')

Filas completamente duplicadas: 126


## Exploración de variables categóricas

Revisamos los valores únicos de cada variable categórica para detectar inconsistencias de
escritura (mayúsculas/minúsculas, abreviaturas, códigos, anglicismos, errores de tipeo).

In [12]:
# subscription_plan: valores únicos y su frecuencia
print('=== subscription_plan ===')
print(df['subscription_plan'].value_counts())

=== subscription_plan ===
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64


In [13]:
# country: valores únicos y su frecuencia
print('=== country ===')
print(df['country'].value_counts())

=== country ===
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
argentina       16
MEX             16
Chile           16
PER             16
Peru            15
BRA             15
chile           15
Mexico          15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64


In [14]:
# favorite_genre: valores únicos y su frecuencia
print('=== favorite_genre ===')
print(df['favorite_genre'].value_counts(dropna=False))

=== favorite_genre ===
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Romance        1090
Thriller       1090
Acción         1082
Crime          1067
None            240
Action           22
COMEDIA          19
Crimen           18
Romance          17
CRIME            17
Documentary      16
DRAMA            16
Comedia          15
DOC              15
ROMANCE          14
ACCIÓN           14
THRILLER         14
comedy           12
Thriller         12
romance          12
documental       10
drama            10
accion            8
Drama             7
thriler           6
crime             5
Name: count, dtype: int64


## Exploración de variables numéricas

Usamos `describe()` para detectar valores fuera de rango razonable (edades negativas, tiempos
de visualización imposibles, etc.).

In [15]:
# age: estadísticas descriptivas
print('=== age ===')
print(df['age'].describe())

=== age ===
count    8160.000000
mean       34.096814
std        14.511304
min        -5.000000
25%        25.000000
50%        33.000000
75%        42.000000
max       150.000000
Name: age, dtype: float64


In [18]:
# Cantidad de edades fuera de un rango admisible (referencia: 0 a 100 años)
edad_invalida = df[(df['age'] < 0) | (df['age'] > 100)]
print(f'Registros con edad fuera de rango admisible (0-100): {len(edad_invalida)}')
edad_invalida[['user_id', 'age']].head(10)

Registros con edad fuera de rango admisible (0-100): 74


,user_id,age
194,10194,130
324,10324,130
426,10426,150
442,10442,-5
529,10529,130
573,10573,-5
640,10640,150
655,10655,130
751,10751,150
1028,11028,150


In [19]:
# monthly_watch_time_mins: estadísticas descriptivas
print('=== monthly_watch_time_mins ===')
print(df['monthly_watch_time_mins'].describe())

=== monthly_watch_time_mins ===
count     7967.000000
mean      1107.346153
std       5310.442604
min       -120.000000
25%        489.200000
50%        757.400000
75%       1045.700000
max      99999.000000
Name: monthly_watch_time_mins, dtype: float64


In [20]:
# Registros con tiempo de visualización negativo o extremadamente alto
tiempo_invalido = df[(df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] > 5000)]
print(f'Registros con monthly_watch_time_mins fuera de rango plausible: {len(tiempo_invalido)}')
tiempo_invalido[['user_id', 'monthly_watch_time_mins']].head(10)

Registros con monthly_watch_time_mins fuera de rango plausible: 80


,user_id,monthly_watch_time_mins
176,10176,50000.0
430,10430,99999.0
443,10443,-120.0
481,10481,50000.0
588,10588,99999.0
797,10797,-1.0
1072,11072,99999.0
1126,11126,-1.0
1186,11186,-120.0
1250,11250,99999.0


In [21]:
# customer_support_tickets: estadísticas descriptivas
print('=== customer_support_tickets ===')
print(df['customer_support_tickets'].describe())

=== customer_support_tickets ===
count    8160.000000
mean        1.800980
std        11.334969
min        -1.000000
25%         0.000000
50%         1.000000
75%         1.000000
max       150.000000
Name: customer_support_tickets, dtype: float64


In [22]:
# Registros con cantidad de tickets negativa o extremadamente alta
tickets_invalido = df[(df['customer_support_tickets'] < 0) | (df['customer_support_tickets'] > 50)]
print(f'Registros con customer_support_tickets fuera de rango plausible: {len(tickets_invalido)}')
tickets_invalido[['user_id', 'customer_support_tickets']].head(10)

Registros con customer_support_tickets fuera de rango plausible: 96


,user_id,customer_support_tickets
0,10000,99
300,10300,150
366,10366,99
378,10378,-1
578,10578,150
609,10609,150
682,10682,-1
811,10811,150
1052,11052,150
1099,11099,-1


## Exploración de `last_login_date`

Revisamos el rango de fechas y detectamos registros que no pueden convertirse a un formato de
fecha válido.

In [23]:
# Intentamos convertir la columna a fecha; los valores no parseables quedan como NaT
fechas = pd.to_datetime(df['last_login_date'], errors='coerce')

print(f'Fecha mínima: {fechas.min()}')
print(f'Fecha máxima: {fechas.max()}')

no_parseables = df['last_login_date'].notna().sum() - fechas.notna().sum()
print(f'Valores no nulos que no se pudieron convertir a fecha: {no_parseables}')

Fecha mínima: 2018-01-01 00:00:00
Fecha máxima: 2029-01-01 00:00:00
Valores no nulos que no se pudieron convertir a fecha: 449


In [24]:
# Registros con fecha de login posterior a la fecha actual del proyecto (fechas "futuras")
fecha_corte = pd.Timestamp('2026-06-29')
fechas_futuras = df[fechas > fecha_corte]
print(f'Registros con last_login_date posterior a {fecha_corte.date()}: {len(fechas_futuras)}')
fechas_futuras[['user_id', 'last_login_date']].head(10)

Registros con last_login_date posterior a 2026-06-29: 15


,user_id,last_login_date
58,10058,2029-01-01
69,10069,2029-01-01
162,10162,2029-01-01
737,10737,2029-01-01
908,10908,2029-01-01
1011,11011,2029-01-01
1580,11580,2029-01-01
1712,11712,2029-01-01
2272,12272,2029-01-01
2973,12973,2029-01-01
